# BPI2019 — 02 · Classification feature re-engineering (EDA)

**Goal:** predict the new **k=6** behavioural cohorts (from `01_clustering_feature_eda`) using **static
context only**, with **native-categorical** models (no one-hot). Compare **XGBoost vs CatBoost**, aim
for a **decent F1** (report macro + weighted vs class-prior baselines). Bounded — no endless tuning.

Context features are fed directly as categories (reasonable cardinality): `item_category`, `item_type`,
`spend_area` (folded to top values covering ~80%, cap 10, + `Others`), `spend_classification`,
`document_type`, `company`, `gr_based_inv_verif`, `goods_receipt`, plus numeric `case_value_eur_log`.

## 1. Load cohorts + build/inspect context features

In [6]:
import os, json, warnings, time

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.dummy import DummyClassifier
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
KEY = "case:concept:name"
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))  # repository root (parent of notebooks/)
INTERIM = os.path.join(REPO, "data", "interim")
OUT = os.path.join(REPO, "artifacts")

labels = pd.read_csv(os.path.join(OUT, "cohort_labels.csv"))
cases = pd.read_parquet(os.path.join(INTERIM, "bpi2019_cases.parquet"))
stat = pd.read_csv(os.path.join(INTERIM, "bpi2019_static.csv"))

RAW = {
    "Item Category": "item_category",
    "Item Type": "item_type",
    "Spend area text": "spend_area",
    "Sub spend area text": "sub_spend_area",  # more granular spend taxonomy (136 -> folded)
    "Spend classification text": "spend_classification",
    "Document Type": "document_type",
    "Company": "company",
}
df = labels.merge(cases[[KEY] + list(RAW)].rename(columns=RAW), on=KEY, how="left")
sidx = stat.set_index(KEY).reindex(df[KEY]).reset_index(drop=True)
df["gr_based_inv_verif"] = np.where(sidx["gr_based_inv_verif"] == 1, "Yes", "No")
df["goods_receipt"] = np.where(sidx["goods_receipt"] == 1, "Yes", "No")
df["case_value_eur_log"] = sidx["case_value_eur_log"].astype(float).values

CATS = list(RAW.values()) + ["gr_based_inv_verif", "goods_receipt"]
for c in CATS:
    df[c] = df[c].astype(str).str.strip().replace({"": "Unknown", "nan": "Unknown"})


def reduce_card(s, cap=12, cover=0.85):
    vc = s.value_counts()
    if len(vc) <= cap:
        return s
    frac = vc.cumsum() / vc.sum()
    keep = set(vc.index[: min(cap, int((frac.values < cover).sum()) + 1)])
    return s.where(s.isin(keep), "Others")


for c in CATS:
    before = df[c].nunique()
    df[c] = reduce_card(df[c])
    if df[c].nunique() != before:
        print(f"  {c}: {before} -> {df[c].nunique()} categories (+Others)")

FEATURES = CATS + ["case_value_eur_log"]
y = df["cohort"].astype(int)
print("\ncohorts:", dict(y.value_counts().sort_index()))
print("cardinality:", {c: df[c].nunique() for c in CATS})

  spend_area: 21 -> 5 categories (+Others)
  sub_spend_area: 136 -> 13 categories (+Others)

cohorts: {0: np.int64(147762), 1: np.int64(8751), 2: np.int64(14492), 3: np.int64(28677), 4: np.int64(846), 5: np.int64(51206)}
cardinality: {'item_category': 4, 'item_type': 6, 'spend_area': 5, 'sub_spend_area': 13, 'spend_classification': 4, 'document_type': 3, 'company': 4, 'gr_based_inv_verif': 2, 'goods_receipt': 2}


## 2. EDA — which context features associate with the cohorts (Cramér's V)

In [7]:
from sklearn.feature_selection import mutual_info_classif


def cramers_v(a, b):
    tab = pd.crosstab(a, b)
    chi2 = chi2_contingency(tab)[0]
    n = tab.values.sum()
    r, k = tab.shape
    return float(np.sqrt((chi2 / n) / max(min(r - 1, k - 1), 1)))


assoc = pd.Series({c: cramers_v(df.cohort, df[c]) for c in CATS}).sort_values(
    ascending=False
)
# association of value magnitude with cohort: correlation ratio (eta^2) of case_value by cohort
gm = df.case_value_eur_log.mean()
sst = ((df.case_value_eur_log - gm) ** 2).sum()
ssb = sum(
    len(v) * (v.mean() - gm) ** 2 for _, v in df.case_value_eur_log.groupby(df.cohort)
)

# mutual information (nats) — captures non-monotonic signal a single tree split can exploit
Xmi = pd.DataFrame({c: df[c].astype("category").cat.codes for c in CATS})
Xmi["case_value_eur_log"] = df["case_value_eur_log"].values
mi = pd.Series(
    mutual_info_classif(
        Xmi, y, discrete_features=[True] * len(CATS) + [False], random_state=SEED
    ),
    index=Xmi.columns,
).sort_values(ascending=False)

summary = pd.DataFrame(
    {
        "cramers_v": assoc.reindex(CATS),
        "mutual_info": mi.reindex(CATS),
        "cardinality": pd.Series({c: df[c].nunique() for c in CATS}),
    }
).sort_values("mutual_info", ascending=False)
print("Context feature EDA (association + information + cardinality):")
print(summary.round(3).to_string())
print(
    f"\ncase_value_eur_log  eta^2 = {ssb/sst:.3f} | mutual_info = {mi['case_value_eur_log']:.3f}"
)
print(f"\nTop context feature by MI: {mi.idxmax()}  ({mi.max():.3f} nats)")

Context feature EDA (association + information + cardinality):
                      cramers_v  mutual_info  cardinality
item_type                 0.390        0.191            6
item_category             0.428        0.160            4
sub_spend_area            0.242        0.075           13
spend_area                0.163        0.045            5
gr_based_inv_verif        0.343        0.029            2
spend_classification      0.113        0.019            4
document_type             0.214        0.018            3
company                   0.044        0.002            4
goods_receipt             0.073        0.002            2

case_value_eur_log  eta^2 = 0.223 | mutual_info = 0.237

Top context feature by MI: case_value_eur_log  (0.237 nats)


## 3. XGBoost vs CatBoost (native categorical, balanced)

In [8]:
Xtr, Xte, ytr, yte = train_test_split(
    df[FEATURES], y, test_size=0.3, random_state=SEED, stratify=y
)
f1b = max(
    f1_score(
        yte,
        DummyClassifier(strategy=s, random_state=SEED).fit(Xtr, ytr).predict(Xte),
        average="macro",
    )
    for s in ["stratified", "most_frequent"]
)

# --- XGBoost: category dtype + enable_categorical + sqrt-balanced weights ---
Xtr_x = Xtr.copy()
Xte_x = Xte.copy()
for c in CATS:
    Xtr_x[c] = Xtr_x[c].astype("category")
    Xte_x[c] = pd.Categorical(Xte_x[c], categories=Xtr_x[c].cat.categories)
wtr = np.sqrt(compute_sample_weight("balanced", ytr))
t = time.time()
xgbc = xgb.XGBClassifier(
    objective="multi:softmax",
    tree_method="hist",
    enable_categorical=True,
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    n_jobs=-1,
    random_state=SEED,
    eval_metric="mlogloss",
)
xgbc.fit(Xtr_x, ytr, sample_weight=wtr)
px = xgbc.predict(Xte_x)
xgb_macro, xgb_w = f1_score(yte, px, average="macro"), f1_score(
    yte, px, average="weighted"
)
print(
    f"XGBoost : acc {accuracy_score(yte, px):.3f} | macro-F1 {xgb_macro:.3f} | weighted-F1 {xgb_w:.3f}  [{time.time()-t:.0f}s]"
)

# --- CatBoost: native cat_features + auto balanced class weights ---
cat_idx = [FEATURES.index(c) for c in CATS]
t = time.time()
cbc = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=SEED,
    verbose=False,
)
cbc.fit(Xtr, ytr, cat_features=cat_idx)
pc = cbc.predict(Xte).ravel().astype(int)
cb_macro, cb_w = f1_score(yte, pc, average="macro"), f1_score(
    yte, pc, average="weighted"
)
print(
    f"CatBoost: acc {accuracy_score(yte, pc):.3f} | macro-F1 {cb_macro:.3f} | weighted-F1 {cb_w:.3f}  [{time.time()-t:.0f}s]"
)
print(f"\nbaseline macro-F1 = {f1b:.3f}")
print("\nBetter model per-class report:")
best_pred = px if xgb_macro >= cb_macro else pc
print(classification_report(yte, best_pred, digits=3))

XGBoost : acc 0.655 | macro-F1 0.492 | weighted-F1 0.628  [18s]
CatBoost: acc 0.470 | macro-F1 0.430 | weighted-F1 0.508  [325s]

baseline macro-F1 = 0.169

Better model per-class report:
              precision    recall  f1-score   support

           0      0.690     0.864     0.767     44329
           1      0.271     0.169     0.208      2625
           2      0.529     0.349     0.421      4348
           3      0.872     0.615     0.721      8603
           4      0.407     0.748     0.527       254
           5      0.416     0.244     0.308     15362

    accuracy                          0.655     75521
   macro avg      0.531     0.498     0.492     75521
weighted avg      0.630     0.655     0.628     75521



## 4. Tune XGBoost + weighting regimes

Bounded `RandomizedSearchCV` (20 draws × 3-fold, scored on `f1_weighted`) on a 50k stratified subsample
for speed; the best params are then refit on the **full** train split. We report two weighting regimes:
- **unweighted** → maximises overall performance (accuracy / weighted-F1);
- **sqrt-balanced** → lifts the rare rework cohorts (macro-F1) at the cost of overall accuracy.

Features stay **static-context only** — adding behavioural features would inflate F1 but break the
decoupling (the classifier's job is to attribute behaviour to *context*, not to peek at behaviour).


In [9]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# tune on a stratified subsample for speed, then refit best params on the full train split
samp = Xtr_x.sample(n=min(50000, len(Xtr_x)), random_state=SEED)
ysamp = ytr.loc[samp.index]

param_dist = {
    "n_estimators": [300, 500, 700],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1, 0.15],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "reg_lambda": [1, 3, 5],
    "gamma": [0.0, 0.5, 1.0],
}
base = xgb.XGBClassifier(
    objective="multi:softmax",
    tree_method="hist",
    enable_categorical=True,
    n_jobs=-1,
    random_state=SEED,
    eval_metric="mlogloss",
)
search = RandomizedSearchCV(
    base,
    param_dist,
    n_iter=20,
    scoring="f1_weighted",
    cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
    random_state=SEED,
    n_jobs=1,
    verbose=0,
)
t = time.time()
search.fit(samp, ysamp)
best_params = search.best_params_
print(f"tuned in {time.time()-t:.0f}s | best CV f1_weighted = {search.best_score_:.3f}")
print("best params:", best_params)

tuned in 219s | best CV f1_weighted = 0.610
best params: {'subsample': 0.85, 'reg_lambda': 3, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0.0, 'colsample_bytree': 0.7}


In [11]:
def fit_eval(weight_mode):
    m = xgb.XGBClassifier(
        objective="multi:softmax",
        tree_method="hist",
        enable_categorical=True,
        n_jobs=-1,
        random_state=SEED,
        eval_metric="mlogloss",
        **best_params,
    )
    sw = (
        np.sqrt(compute_sample_weight("balanced", ytr))
        if weight_mode == "balanced"
        else None
    )
    m.fit(Xtr_x, ytr, sample_weight=sw)
    p = m.predict(Xte_x)
    return {
        "model": m,
        "pred": p,
        "acc": accuracy_score(yte, p),
        "macro": f1_score(yte, p, average="macro"),
        "weighted": f1_score(yte, p, average="weighted"),
    }


results = {tag: fit_eval(tag) for tag in ["unweighted", "balanced"]}
for tag, r in results.items():
    print(
        f"tuned XGB [{tag:10s}]  acc {r['acc']:.3f} | macro-F1 {r['macro']:.3f} | weighted-F1 {r['weighted']:.3f}"
    )
print(f"\nbaseline macro-F1 = {f1b:.3f}")

overall = (
    "unweighted"
    if results["unweighted"]["weighted"] >= results["balanced"]["weighted"]
    else "balanced"
)
print(f"\nPer-class report — overall-optimised model ('{overall}'):")
print(classification_report(yte, results[overall]["pred"], digits=3))
print(f"Per-class report — macro-optimised model ('balanced'):")
print(classification_report(yte, results["balanced"]["pred"], digits=3))

tuned XGB [unweighted]  acc 0.677 | macro-F1 0.430 | weighted-F1 0.602
tuned XGB [balanced  ]  acc 0.655 | macro-F1 0.492 | weighted-F1 0.629

baseline macro-F1 = 0.169

Per-class report — overall-optimised model ('balanced'):
              precision    recall  f1-score   support

           0      0.690     0.863     0.767     44329
           1      0.280     0.168     0.210      2625
           2      0.532     0.349     0.421      4348
           3      0.870     0.615     0.721      8603
           4      0.407     0.748     0.527       254
           5      0.415     0.246     0.309     15362

    accuracy                          0.655     75521
   macro avg      0.532     0.498     0.492     75521
weighted avg      0.630     0.655     0.629     75521

Per-class report — macro-optimised model ('balanced'):
              precision    recall  f1-score   support

           0      0.690     0.863     0.767     44329
           1      0.280     0.168     0.210      2625
           2

In [13]:
# --- Experiment: does adding vendor identity (a static context attribute) break the ceiling? ---
# Vendor is known upfront (not behavioural), so it does not leak execution. It is higher-card and
# closer to memorisation, so we fold to the top vendors + Others and report the lift separately.
dfx = df.copy()
dfx["vendor"] = (
    cases.set_index(KEY)["Vendor"].reindex(dfx[KEY]).astype(str).str.strip().values
)
dfx["vendor"] = reduce_card(dfx["vendor"], cap=40, cover=0.60)
CATS_V = CATS + ["vendor"]
FEAT_V = CATS_V + ["case_value_eur_log"]
XtrV, XteV, _, _ = train_test_split(
    dfx[FEAT_V], y, test_size=0.3, random_state=SEED, stratify=y
)
for c in CATS_V:
    XtrV[c] = XtrV[c].astype("category")
    XteV[c] = pd.Categorical(XteV[c], categories=XtrV[c].cat.categories)
print(f"vendor folded to {dfx['vendor'].nunique()} categories (+Others)")
for tag in ["unweighted", "balanced"]:
    m = xgb.XGBClassifier(
        objective="multi:softmax",
        tree_method="hist",
        enable_categorical=True,
        n_jobs=-1,
        random_state=SEED,
        eval_metric="mlogloss",
        **best_params,
    )
    sw = np.sqrt(compute_sample_weight("balanced", ytr)) if tag == "balanced" else None
    m.fit(XtrV, ytr, sample_weight=sw)
    p = m.predict(XteV)
    print(
        f"+vendor XGB [{tag:10s}]  acc {accuracy_score(yte, p):.3f} | macro-F1 {f1_score(yte, p, average='macro'):.3f} | weighted-F1 {f1_score(yte, p, average='weighted'):.3f}"
    )

vendor folded to 41 categories (+Others)
+vendor XGB [unweighted]  acc 0.689 | macro-F1 0.466 | weighted-F1 0.638
+vendor XGB [balanced  ]  acc 0.663 | macro-F1 0.520 | weighted-F1 0.654


## 5. Pick the model + save


In [14]:
winner = "XGBoost" if xgb_macro >= cb_macro else "CatBoost"
tuned_overall = results[overall]
tuned_bal = results["balanced"]
print(f"Untuned {winner}: macro-F1 {max(xgb_macro, cb_macro):.3f} (baseline {f1b:.3f})")
print(
    f"Tuned XGB overall-optimised: acc {tuned_overall['acc']:.3f} | weighted-F1 {tuned_overall['weighted']:.3f} | macro-F1 {tuned_overall['macro']:.3f}"
)
print(
    f"Tuned XGB macro-optimised  : acc {tuned_bal['acc']:.3f} | weighted-F1 {tuned_bal['weighted']:.3f} | macro-F1 {tuned_bal['macro']:.3f}"
)
json.dump(
    {
        "target_k": int(y.nunique()),
        "features": FEATURES,
        "categorical": CATS,
        "untuned": {
            "xgb": {"macro_f1": round(xgb_macro, 3), "weighted_f1": round(xgb_w, 3)},
            "catboost": {"macro_f1": round(cb_macro, 3), "weighted_f1": round(cb_w, 3)},
        },
        "tuned_best_params": best_params,
        "tuned_cv_f1_weighted": round(float(search.best_score_), 3),
        "tuned_overall_optimised": {
            "weighting": overall,
            "accuracy": round(tuned_overall["acc"], 3),
            "weighted_f1": round(tuned_overall["weighted"], 3),
            "macro_f1": round(tuned_overall["macro"], 3),
        },
        "tuned_macro_optimised": {
            "weighting": "sqrt-balanced",
            "accuracy": round(tuned_bal["acc"], 3),
            "weighted_f1": round(tuned_bal["weighted"], 3),
            "macro_f1": round(tuned_bal["macro"], 3),
        },
        "baseline_macro_f1": round(f1b, 3),
        "winner": winner,
        "assoc_cramers_v": assoc.round(3).to_dict(),
        "mutual_info": mi.round(4).to_dict(),
    },
    open(os.path.join(OUT, "classification_eda.json"), "w"),
    indent=2,
)
print("\nsaved classification_eda.json to", OUT)

Untuned XGBoost: macro-F1 0.492 (baseline 0.169)
Tuned XGB overall-optimised: acc 0.655 | weighted-F1 0.629 | macro-F1 0.492
Tuned XGB macro-optimised  : acc 0.655 | weighted-F1 0.629 | macro-F1 0.492

saved classification_eda.json to <project root>\artifacts
